# EECE 5155 — Acoustic Monitoring: Initial Data Analysis
**Student:** Aiisha Matsungo | **NUID:** 002530298

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/one_hour_acoustic.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"Time range: {df['timestamp'].min()} to {df['timestamp'].max()}")
df.head()

## Descriptive Statistics per Zone

In [ ]:
stats = df.groupby('zone_id')['dba_spl'].describe().round(2)
print(stats)

## dBA SPL Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for zone, grp in df.groupby('zone_id'):
    ax.plot(grp['timestamp'], grp['dba_spl'], label=zone, alpha=0.7)
ax.axhline(45, color='red', linestyle='--', label='45 dBA threshold')
ax.set_xlabel('Time')
ax.set_ylabel('dBA SPL')
ax.set_title('Acoustic Monitoring — 1-Hour Collection')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('figures/dba_over_time.png', dpi=150)
plt.show()
print("Saved: figures/dba_over_time.png")

## Violation Rate per Zone

In [ ]:
violation_rate = df.groupby('zone_id')['status'].apply(
    lambda x: (x == 'violation').sum() / len(x) * 100
).round(1)
print("Violation rate (%):")
print(violation_rate)

violation_rate.plot(kind='bar', color='salmon', edgecolor='black')
plt.title('Violation Rate by Zone (%)')
plt.ylabel('% readings above threshold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('figures/violation_rate.png', dpi=150)
plt.show()

## Observations
- `snell_group_room` has highest mean dBA (48.7) and most violations due to relaxed 55 dBA threshold being frequently approached
- `snell_silent_study` shows lowest mean (39.9 dBA) but highest spike amplitude (max 62.3 dBA) — small room reflections amplify transient events
- `snell_reading_hall` is smoothest profile (std=3.4) — large open space dissipates sound uniformly
- Light level positively correlates with sound level (both occupancy-driven)
- No missing data — all 363 records recovered cleanly from InfluxDB